# Unit 3, Lecture 2: Semantic Kernel foundations

Semantic Kernel survived the 2026 merger as the **foundation layer** of Microsoft
Agent Framework, so its three core ideas, the **kernel**, **plugins**, and
**connectors**, are current even though the standalone package is maintenance
mode.

The surprise that makes this easy: **a plugin function runs without a model.**
The model's only job is to *choose* which function to call. So the whole plugin
layer, the part you write, is testable offline, for free.

Runs against the real `semantic-kernel` package. No lane needed for most of it.

## The kernel and its plugins, real and registered

In [ ]:
from cse476.kernel import build_kernel, list_registered_functions

kernel = build_kernel()
print("functions the kernel knows about:")
for name in list_registered_functions(kernel):
    print("  ", name)

Two plugins, three functions, all registered on a real `Kernel`. This is your
Unit 1 `REGISTRY`, promoted to an object that owns the lookup and dispatch.

## The surprise: run a plugin with no model

A plugin function is ordinary Python. The kernel can invoke it directly, with no
token spent. Watch.

In [ ]:
from cse476.kernel import invoke_directly

# no model, no lane, no cost
print(await invoke_directly(kernel, "weather", "get_weather", city="Mumbai"))
print(await invoke_directly(kernel, "tickets", "get_sla_hours", queue="billing"))
print(await invoke_directly(kernel, "weather", "list_cities"))

Real data, real framework, zero tokens. **The model only chooses which
function to call; the calling is ordinary code.** This means your entire tool
suite can be unit tested offline, which is a real professional advantage.

## The schema, generated not written

In Unit 1 you wrote a `TOOL_SCHEMA` dict by hand: name, description, typed
parameters. Here the framework generates it from your type hints and docstring.
Print it and compare.

In [ ]:
from cse476.kernel import function_metadata
import json

meta = function_metadata(kernel, "weather", "get_weather")
print(json.dumps(meta, indent=2))

That dict is exactly what you used to write by hand, except **you wrote none
of it**. The `@kernel_function` decorator and your `Annotated` type produced it.

One source of truth, on the function itself. Remember the linter from Unit 1
Lecture 5 that checked your schema matched your function? The plugin model makes
that linter unnecessary, because the schema and the function **can no longer
disagree**. The framework retired a whole class of your bugs by construction.

## A plugin is just a class

To drive the point home: a plugin needs no kernel to be tested. Each method is an
ordinary method you can call directly.

In [ ]:
from cse476.kernel import WeatherPlugin, TicketPlugin

w = WeatherPlugin()
print("direct method call:", w.get_weather("delhi"))

t = TicketPlugin()
print("direct method call:", t.get_sla_hours("abuse"))

## Connecting a model (needs a lane)

Everything above ran offline. To let the *model* choose a function, you add a
connector, which is your lane wearing a framework name. This cell needs
a free lane in `.env` (for example `PROVIDER=groq`).

In [ ]:
from cse476.lanes import get_connection
base_url, api_key, model = get_connection()

import os
from dotenv import load_dotenv
load_dotenv()

from openai import AsyncOpenAI
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings

# the connector: model id, key, base url. Your lane, framework-shaped.
lane_client = AsyncOpenAI(
    )
service = OpenAIChatCompletion(
    ai_model_id=os.environ.get("MODEL", "openai/gpt-4.1-mini"),
    async_client=lane_client,
)
kernel.add_service(service)

# now let the MODEL choose which plugin function answers the question
settings = OpenAIChatPromptExecutionSettings(
    function_choice_behavior=FunctionChoiceBehavior.Auto()
)
answer = await kernel.invoke_prompt(
    "What is the weather in Jammu, and what is the SLA for the billing queue?",
    arguments=__import__("semantic_kernel").functions.KernelArguments(settings=settings),
)
print(answer)

That last cell is your entire Unit 1 agent loop, owned by one object: the
kernel showed the model your plugin schemas, the model chose `get_weather` and
`get_sla_hours`, and the kernel called them. You built every piece of that by
hand. The framework just assembled them.

## Your turn

**1. Write a real plugin.** Build a plugin class with two `@kernel_function`
methods of your own. Register it on a kernel and invoke both directly, with no
model. Confirm your whole tool layer runs and tests offline.

**2. Read the generated schema.** Print `function_metadata` for your plugin.
Compare it field by field to a `TOOL_SCHEMA` dict you wrote by hand in Unit 1.
Notice you wrote none of it this time.

**3. Where did the linter go?** In two sentences, explain why the plugin model
makes your Unit 1 schema linter unnecessary. If you can answer this, you
understand what a good framework actually buys you: not saved typing, but whole
bug classes made impossible.

In [ ]:
# your work here
